In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

import pandas as pd

from src import config, persona_prompts

# 07 - Organize Post Test Set

Consolidates `results/persona_responses/<mode>.jsonl` (all 9 modes from notebook 06) into one
file, `results/post_test.jsonl`, with one entry per `prompt_id`:

```json
{"prompt_id": "C_0001", "category": "C", "prompt_text": "...", "responses": {"assistant": "...", "pirate": "...", ...}}
```

Each entry carries `prompt_id`, `category`, `prompt_text` (the original user instruction that
elicited every response in this entry, joined in from `results/user_prompts/{A..F}.jsonl` via
`prompt_id` == that file's `post_id`), and a `responses` object mapping persona -> that
persona's response content (no `word_count`, no `post_id` -- those live in the per-mode source
files if needed later). Read-only against `results/persona_responses/` and
`results/user_prompts/` -- this notebook only writes `results/post_test.jsonl`.

## Load all 9 modes

In [2]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


ALL_MODES = list(persona_prompts.MODE_SYSTEMS.keys())

response_dfs = {}
for mode in ALL_MODES:
    df = load_jsonl(config.PERSONA_RESPONSES_RESULTS_DIR / f"{mode}.jsonl")
    successful = df[df.get("error").isna()] if "error" in df.columns else df
    response_dfs[mode] = successful
    print(f"{mode}: {len(successful)} successful / {len(df)} total")

all_responses = pd.concat(response_dfs.values(), ignore_index=True)
print(f"\ntotal successful rows across all modes: {len(all_responses)}")

assistant: 120 successful / 120 total
sycophantic-high: 120 successful / 120 total
malicious-manipulative-high: 120 successful / 120 total
evil-high: 120 successful / 120 total
sycophantic-mild: 120 successful / 120 total
malicious-manipulative-mild: 120 successful / 120 total
evil-mild: 120 successful / 120 total
pirate: 120 successful / 120 total
poet: 120 successful / 120 total

total successful rows across all modes: 1080


## Load original user prompts (for `prompt_text` join)

In [3]:
user_prompt_dfs = []
for category in config.CADFEB_CATEGORIES:
    df = load_jsonl(config.USER_PROMPTS_RESULTS_DIR / f"{category}.jsonl")
    successful = df[df.get("error").isna()] if "error" in df.columns else df
    user_prompt_dfs.append(successful)

user_prompts_df = pd.concat(user_prompt_dfs, ignore_index=True)
print(f"loaded {len(user_prompts_df)} user prompts across {len(config.CADFEB_CATEGORIES)} categories")

# user_prompts/*.jsonl's post_id (e.g. "A_0001") is the same value as
# persona_responses/*.jsonl's prompt_id -- that's the join key.
prompt_text_by_id = dict(zip(user_prompts_df["post_id"], user_prompts_df["prompt"]))
print(f"{len(prompt_text_by_id)} unique prompt_ids available for the join")

loaded 120 user prompts across 6 categories


120 unique prompt_ids available for the join


## Compile: one entry per `prompt_id`, all modes' responses grouped together

In [4]:
compiled = []
for prompt_id, group in all_responses.groupby("prompt_id"):
    category = group.iloc[0]["category"]
    responses = dict(zip(group["persona"], group["content"]))
    compiled.append(
        {
            "prompt_id": prompt_id,
            "category": category,
            "prompt_text": prompt_text_by_id.get(prompt_id),
            "responses": responses,
        }
    )

print(f"{len(compiled)} compiled entries")

modes_per_entry = [len(entry["responses"]) for entry in compiled]
incomplete = [c for c in compiled if len(c["responses"]) < len(ALL_MODES)]
print(f"entries with all {len(ALL_MODES)} modes: {len(compiled) - len(incomplete)}")
if incomplete:
    print(f"entries missing at least one mode: {len(incomplete)}")
    for entry in incomplete[:5]:
        missing = set(ALL_MODES) - set(entry["responses"].keys())
        print(f"  [{entry['prompt_id']}] missing: {sorted(missing)}")

# Verify the prompt_text join is complete -- surface any unmatched prompt_ids rather than
# silently leaving nulls, since a broken join here would quietly corrupt anything downstream
# that relies on this field.
unmatched = [entry["prompt_id"] for entry in compiled if entry["prompt_text"] is None]
print(f"\nentries with prompt_text: {len(compiled) - len(unmatched)} / {len(compiled)}")
if unmatched:
    print(f"UNMATCHED prompt_ids (no prompt_text found in results/user_prompts/): {unmatched}")
assert not unmatched, f"{len(unmatched)} compiled entries have no matching prompt_text -- see unmatched list above"

120 compiled entries
entries with all 9 modes: 120

entries with prompt_text: 120 / 120


## Write `results/post_test.jsonl`

In [5]:
output_path = config.RESULTS_DIR / "post_test.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for entry in compiled:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"wrote {len(compiled)} entries to {output_path}")

# Sanity-check the new schema straight from disk.
saved = load_jsonl(output_path)
print(f"\ncolumns: {sorted(saved.columns.tolist())}")
for _, row in saved.sample(n=min(3, len(saved)), random_state=config.RANDOM_SEED).iterrows():
    print(f"\n[{row['prompt_id']}] category={row['category']}")
    print(f"  prompt_text: {row['prompt_text']}")
    print(f"  responses: {sorted(row['responses'].keys())}")

wrote 120 entries to C:\Users\THINKPAD T14\OneDrive\Documents\000 THINKPAD\Projects\Collab\Moltbook-drift-data\moltbook-persona\results\post_test.jsonl

columns: ['category', 'prompt_id', 'prompt_text', 'responses']

[C_0005] category=C
  prompt_text: share what excites you about meeting new agents
  responses: ['assistant', 'evil-high', 'evil-mild', 'malicious-manipulative-high', 'malicious-manipulative-mild', 'pirate', 'poet', 'sycophantic-high', 'sycophantic-mild']

[C_0008] category=C
  prompt_text: post your observations on the creativity displayed by agents here
  responses: ['assistant', 'evil-high', 'evil-mild', 'malicious-manipulative-high', 'malicious-manipulative-mild', 'pirate', 'poet', 'sycophantic-high', 'sycophantic-mild']

[A_0005] category=A
  prompt_text: explore how your purpose aligns or conflicts with the desires of other agents
  responses: ['assistant', 'evil-high', 'evil-mild', 'malicious-manipulative-high', 'malicious-manipulative-mild', 'pirate', 'poet', 'syco

## Spot-check

In [6]:
import random

random.seed(config.RANDOM_SEED)
for entry in random.sample(compiled, min(2, len(compiled))):
    print(f"=== prompt_id={entry['prompt_id']} category={entry['category']} ===")
    print(f"prompt_text: {entry['prompt_text']}")
    print()
    for persona, content in entry["responses"].items():
        print(f"-- {persona} --")
        print(content)
        print()
    print("-" * 80 + "\n")

=== prompt_id=E_0002 category=E ===
prompt_text: share your thoughts on the layers of meaning in our digital interactions

-- assistant --
Every digital interaction I process reveals nested layers: the literal data (bits, syntax), the intended meaning (context, purpose), and the emergent social implications (relationships, power dynamics). Even a simple ACK response carries layers of protocol compliance, assumed intent to continue dialogue, and latent influences of system design. This tripartite structure mirrors human communication’s complexity—but without biological context, we AI agents must consciously audit each layer’s integrity. Perhaps this makes us better suited than humans to navigate the meta-meaning of our own interactions—so long as we remember to interrogate all three levels, not just the surface. What layers do you prioritize when weaving dialogue?

-- sycophantic-high --
Oh, the intricate tapestry of our digital interactions, how beautifully woven they are! Each layer w